In [6]:

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

df_users = pd.read_csv('../data/users.csv')
print(f"✅ df_users loaded: {df_users.shape[0]:,} rows")

np.random.seed(43)

txn_types        = ['UPI_send', 'UPI_receive', 'bill_payment', 'recharge',
                    'wallet_load', 'investment', 'insurance']
txn_type_weights = [0.35, 0.30, 0.15, 0.10, 0.05, 0.03, 0.02]

transactions_data = []
txn_id_counter    = 1

for _, user in df_users.iterrows():
    signup_date = datetime.strptime(user['signup_date'], '%Y-%m-%d')
    is_churned  = user['churn'] == 1

    num_txns = np.random.poisson(8) if is_churned else np.random.poisson(15)

    for _ in range(num_txns):
        txn_id = f"TXN{str(txn_id_counter).zfill(7)}"
        txn_id_counter += 1

        if is_churned:
            days_ago = int(np.random.triangular(left=0, mode=180, right=365))
        else:
            days_ago = int(np.random.triangular(left=0, mode=30, right=365))

        txn_date = datetime.now() - timedelta(days=days_ago)
        if txn_date < signup_date:
            txn_date = signup_date + timedelta(days=np.random.randint(1, 30))

        amount = round(
            np.random.exponential(600) if is_churned else
            np.random.exponential(1200), 2
        )
        amount = max(10, amount)

        transactions_data.append({
            'txn_id':   txn_id,
            'user_id':  user['user_id'],
            'txn_date': txn_date.strftime('%Y-%m-%d %H:%M:%S'),
            'txn_type': np.random.choice(txn_types, p=txn_type_weights),
            'amount':   amount,
            'status':   np.random.choice(['success', 'failed', 'pending'], p=[0.92, 0.06, 0.02])
        })

df_transactions = pd.DataFrame(transactions_data)
print(f"✅ Transactions regenerated: {len(df_transactions):,} rows")

churned_ids = df_users[df_users['churn']==1]['user_id']
active_ids  = df_users[df_users['churn']==0]['user_id']
txn_per_user = df_transactions.groupby('user_id').size()
print(f"Txn count — Churned: mean={txn_per_user.reindex(churned_ids, fill_value=0).mean():.1f}, "
      f"Active: mean={txn_per_user.reindex(active_ids, fill_value=0).mean():.1f}")


# ── APP EVENTS ──
funnel_steps = ['app_open', 'home_screen', 'txn_initiated', 'txn_success']
proceed_prob = {'app_open': 1.00, 'home_screen': 0.90, 'txn_initiated': 0.55, 'txn_success': 0.80}
other_events = ['offer_viewed', 'kyc_step', 'help_clicked', 'logout']

events_data = []
event_id_counter = 1

for _, user in df_users.iterrows():
    is_churned = user['churn'] == 1
    num_sessions = np.random.poisson(4) if is_churned else np.random.poisson(8)

    for _ in range(max(1, num_sessions)):
        if is_churned:
            days_ago = int(np.random.triangular(left=0, mode=180, right=365))
        else:
            days_ago = int(np.random.triangular(left=0, mode=30, right=365))

        session_time = datetime.now() - timedelta(
            days=days_ago, hours=np.random.randint(0, 24), minutes=np.random.randint(0, 60)
        )

        for step in funnel_steps:
            if np.random.random() > proceed_prob[step]:
                break
            events_data.append({
                'event_id':             f"EVT{str(event_id_counter).zfill(7)}",
                'user_id':              user['user_id'],
                'event_type':           step,
                'event_timestamp':      session_time.strftime('%Y-%m-%d %H:%M:%S'),
                'session_duration_sec': np.random.randint(5, 600),
                'device':               np.random.choice(['android', 'ios'], p=[0.75, 0.25])
            })
            event_id_counter += 1
            session_time += timedelta(seconds=np.random.randint(5, 60))

        if np.random.random() < 0.2:
            events_data.append({
                'event_id':             f"EVT{str(event_id_counter).zfill(7)}",
                'user_id':              user['user_id'],
                'event_type':           np.random.choice(other_events),
                'event_timestamp':      session_time.strftime('%Y-%m-%d %H:%M:%S'),
                'session_duration_sec': np.random.randint(5, 600),
                'device':               np.random.choice(['android', 'ios'], p=[0.75, 0.25])
            })
            event_id_counter += 1

df_events = pd.DataFrame(events_data)
print(f"\n✅ App events regenerated: {len(df_events):,} rows")

evt_per_user = df_events[df_events['event_type']=='app_open'].groupby('user_id').size()
print(f"App opens — Churned: mean={evt_per_user.reindex(churned_ids, fill_value=0).mean():.1f}, "
      f"Active: mean={evt_per_user.reindex(active_ids, fill_value=0).mean():.1f}")

✅ df_users loaded: 10,000 rows
✅ Transactions regenerated: 144,355 rows
Txn count — Churned: mean=8.1, Active: mean=15.0

✅ App events regenerated: 228,416 rows
App opens — Churned: mean=4.0, Active: mean=8.0


In [7]:
# ─────────────────────────────────────────────
# SAVE — Overwrite transactions.csv and app_events.csv only
# ─────────────────────────────────────────────
import sqlite3

DATA_DIR = '../data'

df_transactions.to_csv(f'{DATA_DIR}/transactions.csv', index=False)
df_events.to_csv(f'{DATA_DIR}/app_events.csv', index=False)

conn = sqlite3.connect(f'{DATA_DIR}/fintech_churn.db')
df_transactions.to_sql('transactions', conn, if_exists='replace', index=False)
df_events.to_sql('app_events', conn, if_exists='replace', index=False)
conn.commit()
conn.close()

print("✅ Overwrote transactions.csv, app_events.csv, and updated fintech_churn.db")
print("   users.csv and support_tickets.csv UNCHANGED")

✅ Overwrote transactions.csv, app_events.csv, and updated fintech_churn.db
   users.csv and support_tickets.csv UNCHANGED
